# Class Quality Assessor on Google Colab

Everything runs on Google's machines and the recordings are read **straight from Google Drive**, so nothing big is uploaded. Run the cells top to bottom (Shift+Enter). A GPU is not needed: *Runtime → Change runtime type → CPU* is fine.

**Before you start (once):**
1. 🔑 In the left sidebar open **Secrets** → *Add new secret* → name `GEMINI_API_KEY`, paste your key, and switch on *Notebook access*.
2. In Google Drive open **Shared with me** → right-click *Class Recordings(24-Hour Build Challenge)* → **Organize → Add shortcut → My Drive**.

## 1. Connect Google Drive

In [ ]:
from google.colab import drive, userdata, files
import os, zipfile, glob
drive.mount('/content/drive')
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
print('Key loaded:', bool(os.environ['GEMINI_API_KEY']))

## 2. Get the code (the small ~40 KB code zip, NOT the recordings)
If `class-quality-assessor.zip` is in your My Drive it is used automatically. Otherwise an upload button appears: choose **class-quality-assessor.zip** (about 40 KB). It uploads in a second.

In [ ]:
os.chdir('/content')
zips = glob.glob('/content/drive/MyDrive/class-quality-assessor*.zip') + glob.glob('/content/drive/MyDrive/*/class-quality-assessor*.zip')
if zips:
    src = zips[0]
else:
    up = files.upload()
    src = next(iter(up))
zipfile.ZipFile(src).extractall('/content')
roots = glob.glob('/content/**/calibrate.py', recursive=True)
APP = os.path.dirname(sorted(roots, key=len)[0])
%cd $APP
!ls

## 3. Install (about 1 minute; ffmpeg is already on Colab)

In [ ]:
!pip -q install -r requirements.txt
!ffmpeg -version | head -1

## 4. Find the recordings in Drive
The folder names (excellent, good, average, weak, quality issue) are used as the labels; the `demo` folder is skipped. Each video is matched with its Zoom `.vtt` transcript.

In [ ]:
cands = glob.glob('/content/drive/MyDrive/*Class Recordings*') + glob.glob('/content/drive/MyDrive/*/*lass recordings*')
print(cands)
DATA = os.path.realpath(cands[0])
for p in sorted(glob.glob(DATA + '/**/*', recursive=True)):
    if p.endswith(('.mp4', '.vtt')):
        print(os.path.relpath(p, DATA))

## 5. Test on ONE recording first (2-4 minutes)

In [ ]:
!python calibrate.py --data "$DATA" --only B472 --no-write

## 6. Calibrate on all of them (about 30-45 minutes)
Results are cached while the Colab session stays alive, so if a file fails just run this cell again and it continues.
**When it finishes, copy the table printed at the end and paste it to Claude.**

In [ ]:
!python calibrate.py --data "$DATA"

In [ ]:
# keep the fitted thresholds: put this file in the project folder and commit it
files.download('calibration.json')

## 7. Optional: try the app in your browser
Opens a temporary public link to the app running in this notebook. It stops when the notebook stops, so it is **not** the link to submit (use Hugging Face Spaces for that).

In [ ]:
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared
!nohup streamlit run app.py --server.port 8501 --server.headless true > streamlit.log 2>&1 &
!nohup ./cloudflared tunnel --url http://localhost:8501 > tunnel.log 2>&1 &
import time, re
time.sleep(15)
print(re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', open('tunnel.log').read()))